# 05 · Attention influence and local alignment

This notebook regenerates the results of the paper's attention-influence and local-alignment analyses:

* **Table `tab:ieee-best-subject`**: mean absolute change of MIL-CREDA's logit of ADHD when the 25 % of windows
  with the highest attention, the lowest attention, 25 % at random (ten draws) or the known class-event windows
  (synthetic only) are removed; on ADHD/Control Children for the label-free *cef* pick and the best-test (oracle)
  pick, and on the synthetic pool (24 units, and the 12 units with class effect $d_C \ge 1.5$).
* **The influence numbers of the text**: the matched random baseline of the class-event windows, the share of units
  and subjects where the highest quarter beats a random quarter, the one-sided paired Wilcoxon tests, and the share
  of subjects whose logit moved toward the true class.
* **Figure `fig:ieee-alignment`**: the bag representations of MIL-CREDA on ADHD/Control Children along its second
  phase, trained with and without the local term, with one same-class pair per class and its distances.

**How it works.** Each measurement retrains MIL-CREDA's pick of a unit from its frozen phase-1 base (no search),
checks that the retrained model reproduces the balanced accuracy (and AUC) the pool reported, and only then
measures. Influence is the absolute change of the logit when a set of windows is removed from the bag.


## Setup

The first cell finds the repository (`src/` relative to this notebook). On Kaggle it clones
https://github.com/Daprosero/Domain_Adaptation first.

In [ ]:
import os, sys, subprocess
from pathlib import Path

def find_repo():
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / "src" / "MIL_CREDA_EEG").is_dir():
            return p
    if Path("/kaggle/working").exists():
        dest = Path("/kaggle/working/Domain_Adaptation")
        if not dest.exists():
            subprocess.run(["git", "clone", "--depth", "1", "https://github.com/Daprosero/Domain_Adaptation", str(dest)], check=True)
        return dest
    raise FileNotFoundError("run this notebook from inside the Domain_Adaptation repository")

REPO = find_repo()
sys.path.insert(0, str(REPO / "src"))
from MIL_CREDA_EEG import repro
from MIL_CREDA_EEG.reproduce import stage05_influence as S
print("repository:", REPO)

## Data

* **ADHD/Control Children** comes from the public Kaggle dataset `daprosero/mi-tdah-dataset` (CC0).
  `repro.ieee_root()` uses the mounted copy inside Kaggle (attach the dataset to the notebook) and otherwise
  downloads only the 121 IEEE `.mat` files (about 33 MB) with `kagglehub`.
* **The synthetic units** are simulated on the fly by `eeg_sim` from each unit's key; nothing is downloaded.
* **Frozen inputs** tracked in the repository: the phase-1 bases (`Manifests/oracle_bases`, `Manifests/ieee_pool_bases`)
  and MIL-CREDA's picks (`Manifests/pool_aggregation/milcreda_picks.json`,
  `Manifests/attention_influence/ieee_{cef,oracle}_picks.json`). These picks are the ones the synthetic criteria pool
  (stage 02) and the ADHD pool (stage 04) chose; this stage reads them from the manifests, not from those notebooks.

In [ ]:
print("IEEE ADHD folder:", repro.ieee_root())

## Parameters

Taken from the paper's Kaggle job folders (`ai-*/run-config.json`, branch `ieee-pool-slim`) and the code defaults
those jobs used:

| quantity | value | source |
|---|---|---|
| entrypoints | `attention_influence.run`, `run_ieee(tag="cef"/"oracle")`, `run_alignment` | `run.module` / `run.function` |
| CPU threads per run | 1 | `run.kwargs.threads` |
| windows removed | 25 % (`FRAC`), 10 random draws (`DRAWS`) | `attention_influence.py` |
| phase-2 epochs | 100 (`criteria_pool.EPOCHS2`); alignment snapshots at epochs 0, 50, 100 | code default |
| synthetic evaluation subjects | 120 per unit (`oracle_screen.EVAL_SUBJECTS`) | code default |
| reproduction tolerance | 1e-9 on pool BA (synthetic), 1e-4 on test BA and AUC (ADHD) | `TOLERANCE`, `IEEE_TOLERANCE` |
| work items | 24 synthetic units + 10 ADHD units x (cef, oracle, alignment) = 54 | `S.UNITS` |

**Cost of the full run.** Kaggle measured about 410 s per synthetic unit, 44 s per ADHD influence unit and 118 s per
ADHD alignment unit on one CPU thread: about 3.3 CPU hours in total, no GPU. The paper ran it as 27 Kaggle CPU jobs
(24 synthetic `ai-<unit>` jobs plus `ai-ieee-cef`, `ai-ieee-oracle2` and `ai-ieee-align`), spread over the project's
Kaggle accounts.

In [ ]:
print(S.full_run_cost())
print(len(S.UNITS), "work items, e.g.", S.UNITS[:2], "...", S.UNITS[-1:])

## Split and mode

* `SPLIT = None` runs every item in this account. `SPLIT = (k, n)` runs items k, k+n, k+2n, ... so that n accounts
  (k = 0 .. n-1) share the work; each writes its raw outputs into `Results/paper/05_attention_influence/raw/`.
* `SMOKE` (set the environment variable `REPRO_SMOKE=1`) only proves the wiring: it measures one ADHD unit
  (`ieee-r3h1`, cef pick) in about 15 s into a temporary folder and leaves the stored summary untouched.
* `STRICT = True` keeps the paper's rule: no measurement unless the retrained pick reproduces its reported numbers.
  On a platform other than Kaggle's CPU the 100-epoch training can drift; `STRICT = False` measures anyway and
  records the gap in each output's `reproduction` field.

In [ ]:
SPLIT = None  # or (k, n) to run portion k of n in one Kaggle account
SMOKE = os.environ.get("REPRO_SMOKE") == "1"
STRICT = True
print("portion:", len(repro.portion(S.UNITS, SPLIT)), "items; smoke:", SMOKE)

## Run

Retrains each pick and measures it. The full portion takes hours (see the cost above); the smoke run about a minute
including data loading.

In [ ]:
RAW = S.run(split=SPLIT, smoke=SMOKE, strict=STRICT)
print("raw outputs in", RAW)

## Merge

When several accounts ran portions, copy every account's `raw/` folders into the same
`Results/paper/05_attention_influence/raw/` and run this cell once: it joins them into `summary/`
(influence rows, alignment representations, the statistics and the chosen alignment pair) and lists any item that
is missing, was not measured or did not reproduce. In smoke mode it merges the single item into a temporary folder,
so the stored summary (merged from the paper's own outputs) stays as it is.

In [ ]:
MERGED = S.merge(out=RAW / "summary", raw=RAW) if SMOKE else S.merge()
print(open(MERGED / "status.json").read())

## Render

Draws the figure and writes the table from the stored summary only, with no training. The alignment unit is the
one where the local term most shortened the distance to the nearest source bag (mean over the target test
subjects); per class, the highlighted same-class pair is the one whose final distance shrank most relative to
training without the local term.

In [ ]:
from IPython.display import Image, display, Markdown
out = S.render()
display(Image(filename=out["figures"][1]))
print(open(out["tables"][0]).read())
import json
pair = S.alignment_pair()
print("unit:", pair["unit"])
for c, p in pair["pairs"].items():
    print(["control", "ADHD"][int(c)], "pair:", p["target"], "->", p["source"],
          "with local term", [round(x, 3) for x in p["distance"]["local"]],
          "without", [round(x, 3) for x in p["distance"]["noLocal"]])

## Comparison with the paper

Every number below is quoted from `paper/main.tex` (the table and the paragraphs that cite it) and recomputed from
the stored summary, rounded to the digits the paper prints.

In [ ]:
rows = repro.compare(S.paper_checks())
print(sum(r["within"] for r in rows), "of", len(rows), "within tolerance")